<font size="+3"><strong>8.4. Model Deployment</strong></font>

In [3]:
%load_ext autoreload
%autoreload 2

import os
import sqlite3
from glob import glob

import joblib
import pandas as pd
import requests
from arch.univariate.base import ARCHModelResult
from config import settings
from data import SQLRepository
from IPython.display import VimeoVideo

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Note about Alpha Vantage API access

In this lesson, we will mock the requests library instead of making real requests to the Alpha Vantage API.

Some students have had issues with the live API, especially when requesting the full output size or reaching the daily request limit. Mocking the response allows everyone to work with the same expected data and focus on the lesson content.

Please note that if you open the API link directly in your web browser, you may still see the original API issue. This is expected, because the browser request is not mocked by our local lesson infrastructure.

In [4]:
from mock_alpha import activate_mock
activate_mock()

In [ ]:
VimeoVideo("772219745", h="f3bfda20cd", width=600)

# Model Module

We created a lot of code in the last lesson to building, training, and making predictions with our GARCH(1,1) model. We want this code to be reusable, so let's put it in its own module.

Let's start by instantiating a repository that we'll use for testing our module as we build.

In [5]:
VimeoVideo("772219717", h="8f1afa7919", width=600)

**Task 8.4.1:** Create a `SQLRepository` named `repo`. Be sure that it's attached to a SQLite connection.

- [Open a connection to a SQL database using sqlite3.](../%40textbook/10-databases-sql.ipynb#sqlite3)

In [6]:
connection = sqlite3.connect(settings.db_name, check_same_thread=False)
repo = SQLRepository(connection=connection)

print("repo type:", type(repo))
print("repo.connection type:", type(repo.connection))

repo type: <class 'data.SQLRepository'>
repo.connection type: <class 'sqlite3.Connection'>


Now that we have the `repo` ready, we'll shift to our `model` module and build a `GarchModel` class to hold all our code from the last lesson.

In [7]:
VimeoVideo("772219669", h="1d225ab776", width=600)

**Task 8.4.2:** In the `model` module, create a definition for a `GarchModel` model class. For now, it should only have an `__init__` method. Use the docstring as a guide. When you're done, test your class using the assert statements below.

- [What's a class?](../%40textbook/21-python-object-oriented-programming.ipynb#Classes)
- [Write a class definition in Python.](../%40textbook/21-python-object-oriented-programming.ipynb#Defining-a-Class)
- [Write a class method in Python.](../%40textbook/21-python-object-oriented-programming.ipynb#Methods)
- [What's an assert statement?](../%40textbook/02-python-advanced.ipynb#Testing-Code)
- [Write an assert statement in Python.](../%40textbook/02-python-advanced.ipynb#Testing-Code)

In [8]:
settings.model_directory

'models'

In [10]:
from model import GarchModel

# Instantiate a `GarchModel`
gm_ambuja = GarchModel(ticker="AMBUJACEM.BSE", repo=repo, use_new_data=False)

# Does `gm_ambuja` have the correct attributes?
assert gm_ambuja.ticker == "AMBUJACEM.BSE"
assert gm_ambuja.repo == repo
assert not gm_ambuja.use_new_data
assert gm_ambuja.model_directory == settings.model_directory

In [11]:
gm_ambuja.ticker

'AMBUJACEM.BSE'

In [12]:
VimeoVideo("772219593", h="3f3c401c04", width=600)

**Note about the ticker symbol**

Some videos in this lesson may refer to the ticker `SHOPERSTOP.BSE`. This ticker was previously used to retrieve data from the Alpha Vantage API for Shoppers Stop on the Bombay Stock Exchange.

The notebook has since been updated, and the lesson now uses `SHOPERSTOP.NS` instead. This is the Yahoo Finance/yfinance ticker for Shoppers Stop listed on the National Stock Exchange of India.

Both tickers refer to the same company, but they use different exchange/data-provider conventions:

* `SHOPERSTOP.BSE` was used with Alpha Vantage.
* `SHOPERSTOP.NS` is now used with yfinance/Yahoo Finance.

In other words, the current mock returns yfinance data for SHOPERSTOP.NS, while preserving the structure of a mocked Alpha Vantage API response.

Please follow the ticker used in the updated notebook, even if the video shows the older `SHOPERSTOP.BSE` ticker.


**Task 8.4.3:** Turn your `wrangle_data` function from the last lesson into a method for your `GarchModel` class. When you're done, use the assert statements below to test the method by getting and wrangling data for the department store [Shoppers Stop](https://www.shoppersstop.com/).

- [What's a function?](../%40textbook/02-python-advanced.ipynb#Functions)
- [Write a function in Python.](../%40textbook/02-python-advanced.ipynb#Functions)
- [Write a class method in Python.](../%40textbook/21-python-object-oriented-programming.ipynb#Methods)
- [What's an assert statement?](../%40textbook/02-python-advanced.ipynb#Testing-Code)
- [Write an assert statement in Python.](../%40textbook/02-python-advanced.ipynb#Testing-Code)

In [15]:
# Instantiate `GarchModel`, use new data
model_shop = GarchModel(ticker="SHOPERSTOP.NS", repo=repo, use_new_data=True)

# Check that model doesn't have `data` attribute yet
assert not hasattr(model_shop, "data")

# Wrangle data
model_shop.wrangle_data(n_observations=1000)

# Does model now have `data` attribute?
assert hasattr(model_shop, "data")

# Is the `data` a Series?
assert isinstance(model_shop.data, pd.Series)

# Is Series correct shape?
assert model_shop.data.shape == (1000,)

model_shop.data.head()

date
2022-05-23   -0.344144
2022-05-24   -1.402260
2022-05-25   -2.271280
2022-05-26    2.639010
2022-05-27   -2.338377
Name: return, dtype: float64

In [17]:
VimeoVideo("772219535", h="55fbfdff55", width=600)

**Task 8.4.4:** Using your code from the previous lesson, create a `fit` method for your `GarchModel` class. When you're done, use the code below to test it.

- [Write a class method in Python.](../%40textbook/21-python-object-oriented-programming.ipynb#Methods)
- [What's an assert statement?](../%40textbook/02-python-advanced.ipynb#Testing-Code)
- [Write an assert statement in Python.](../%40textbook/02-python-advanced.ipynb#Testing-Code)<span style='color: transparent; font-size:1%'>WQU WorldQuant University Applied Data Science Lab QQQQ</span>

In [23]:
# Instantiate `GarchModel`, use old data
model_shop = GarchModel(ticker="SHOPERSTOP.NS", repo=repo, use_new_data=False)

# Wrangle data
model_shop.wrangle_data(n_observations=1000)

# Fit GARCH(1,1) model to data
model_shop.fit(p=1, q=1)

# Does `model_shop` have a `model` attribute now?
assert hasattr(model_shop, "model")

# Is model correct data type?
assert isinstance(model_shop.model, ARCHModelResult)

# Does model have correct parameters?
assert model_shop.model.params.index.tolist() == ["mu", "omega", "alpha[1]", "beta[1]"]

# Check model parameters
model_shop.model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                 return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2191.09
Distribution:                  Normal   AIC:                           4390.18
Method:            Maximum Likelihood   BIC:                           4409.81
                                        No. Observations:                 1000
Date:                Thu, Jun 18 2026   Df Residuals:                      999
Time:                        22:45:24   Df Model:                            1
                               Mean Model                               
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu         5.6367e-03  7.036e-02  8.011e-02      0.936 [ -0.132,  0.144]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega          0.9549      2.241      0.426      0.670    [ -3.438,  5.347]
alpha[1]       0.0373  4.678e-02      0.798      0.425 [-5.437e-02,  0.129]
beta[1]        0.7617      0.502      1.517      0.129    [ -0.223,  1.746]
===========================================================================

Covariance estimator: robust
"""

In [24]:
VimeoVideo("772219489", h="3de8abb0e6", width=600)

**Task 8.4.5:** Using your code from the previous lesson, create a `predict_volatility` method for your `GarchModel` class. Your method will need to return predictions as a dictionary, so you'll need to add your `clean_prediction` function as a helper method. When you're done, test your work using the assert statements below.

- [Write a class method in Python.](../%40textbook/21-python-object-oriented-programming.ipynb#Methods)
- [Write a function in Python.](../%40textbook/02-python-advanced.ipynb#Functions)
- [What's an assert statement?](../%40textbook/02-python-advanced.ipynb#Testing-Code)
- [Write an assert statement in Python.](../%40textbook/02-python-advanced.ipynb#Testing-Code)

In [29]:
# Generate prediction from `model_shop`
prediction = model_shop.predict_volatility(horizon=5)

# Is prediction a dictionary?
assert isinstance(prediction, dict)

# Are keys correct data type?
assert all(isinstance(k, str) for k in prediction.keys())

# Are values correct data type?
assert all(isinstance(v, float) for v in prediction.values())

prediction

{'2026-06-08T00:00:00': 2.1716239743470003,
 '2026-06-09T00:00:00': 2.17323998222548,
 '2026-06-10T00:00:00': 2.1745303216513565,
 '2026-06-11T00:00:00': 2.1755607626059357,
 '2026-06-12T00:00:00': 2.176383742103092}

Things are looking good! There are two last methods that we need to add to our `GarchModel` so that we can save a trained model and then load it when we need it. When we learned about saving and loading files in Project 5, we used a context handler. This time, we'll streamline the process using the [joblib library](https://joblib.readthedocs.io/en/latest/). We'll also start writing our filepaths more programmatically using the [os library](https://docs.python.org/3/library/os.html).

In [26]:
VimeoVideo("772219427", h="0dd5731a0d", width=600)

**Task 8.4.6:** Create a `dump` method for your `GarchModel` class. It should save the model assigned to the `model` attribute to the folder specified in your configuration `settings`. Use the docstring as a guide, and then test your work below.

- [Write a class method in Python.](../%40textbook/21-python-object-oriented-programming.ipynb#Methods)
- [Save an object using joblib.](../%40textbook/02-python-advanced.ipynb#Saving-and-Loading-Files-with-joblib)
- [Create a file path using os.](../%40textbook/02-python-advanced.ipynb#Working-with-Filepaths)

In [31]:
model_shop = GarchModel(
    ticker="SHOPERSTOP.NS",
    repo=repo,
    use_new_data=False
)

In [33]:
print(model_shop.model_directory)

models


In [35]:
model_shop.wrangle_data(n_observations=1000)

model_shop.fit(p=1, q=1)

filename = model_shop.dump()

print(filename)

models/2026-06-18T23:07:56.221739_SHOPERSTOP.NS.pkl


In [36]:
# Save `model_shop` model, assign filename
filename = model_shop.dump()

# Is `filename` a string?
assert isinstance(filename, str)

# Does filename include ticker symbol?
assert model_shop.ticker in filename

# Does file exist?
assert os.path.exists(filename)

filename

'models/2026-06-18T23:07:58.373847_SHOPERSTOP.NS.pkl'

In [37]:
VimeoVideo("772219326", h="4e1f9421e4", width=600)

**Task 8.4.7:** Create a `load` function below that will take a ticker symbol as input and return a model. When you're done, use the next cell to load the Shoppers Stop model you saved in the previous task.

- [Handle errors using `try` and `except` blocks in Python.](../%40textbook/02-python-advanced.ipynb#Error-Handling)
- [Create a file path using os.](../%40textbook/02-python-advanced.ipynb#Working-with-Filepaths)
- [Raise an `Exception` in Python.](../%40textbook/02-python-advanced.ipynb#Raising-Errors)

In [39]:
ticker = "SHOPERSTOP.BSE"
pattern = os.path.join(settings.model_directory, f"*{ticker}.pkl")
model_path = sorted(glob(pattern))[-1]
model_path

'models/2025-05-29T13:34:05.187670_SHOPERSTOP.BSE.pkl'

In [ ]:
ticker = "SHOPERSTOP.BSE"
pattern = os.path.join(settings.model_directory, f"*{ticker}.pkl")

try:
    model_path = sorted(glob(pattern))[-1]
except IndexError:
    raise Exception(f"No model trained for '{ticker}'.")

In [46]:
def load(ticker):
    """Load latest model from model directory.

    Parameters
    ----------
    ticker : str
        Ticker symbol for which model was trained.

    Returns
    -------
    `ARCHModelResult`
    """
    # Create pattern for glob search
    pattern = os.path.join(settings.model_directory, f"*{ticker}.pkl")

    # Try to find path of latest model
    try:
        model_path = sorted(glob(pattern))[-1]

    # Handle possible `IndexError`
    except IndexError:
        raise Exception(f"No model trained for '{ticker}'.")

    # Load model
    model = joblib.load(model_path)

    # Return model
    return model

In [45]:
# Assign load output to `model`
model_shop = load(ticker="SHOPERSTOP.NS")

# Does function return an `ARCHModelResult`
assert isinstance(model_shop, ARCHModelResult)

# Check model parameters
model_shop.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                 return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2191.09
Distribution:                  Normal   AIC:                           4390.18
Method:            Maximum Likelihood   BIC:                           4409.81
                                        No. Observations:                 1000
Date:                Thu, Jun 18 2026   Df Residuals:                      999
Time:                        23:07:56   Df Model:                            1
                               Mean Model                               
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu         5.6367e-03  7.036e-02  8.011e-02      0.936 [ -0.132,  0.144]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega          0.9549      2.241      0.426      0.670    [ -3.438,  5.347]
alpha[1]       0.0373  4.678e-02      0.798      0.425 [-5.437e-02,  0.129]
beta[1]        0.7617      0.502      1.517      0.129    [ -0.223,  1.746]
===========================================================================

Covariance estimator: robust
"""

In [47]:
VimeoVideo("772219392", h="deed99bf85", width=600)

**Task 8.4.8:** Transform your `load` function into a method for your `GarchModel` class. When you're done, test the method using the assert statements below.

- [Write a class method in Python.](../%40textbook/21-python-object-oriented-programming.ipynb#Methods)
- [What's an assert statement?](../%40textbook/02-python-advanced.ipynb#Testing-Code)
- [Write an assert statement in Python.](../%40textbook/02-python-advanced.ipynb#Testing-Code)

In [48]:
model_shop = GarchModel(ticker="SHOPERSTOP.NS", repo=repo, use_new_data=False)

# Check that new `model_shop_test` doesn't have model attached
assert not hasattr(model_shop, "model")

# Load model
model_shop.load()

# Does `model_shop_test` have model attached?
assert hasattr(model_shop, "model")

model_shop.model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                 return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2191.09
Distribution:                  Normal   AIC:                           4390.18
Method:            Maximum Likelihood   BIC:                           4409.81
                                        No. Observations:                 1000
Date:                Thu, Jun 18 2026   Df Residuals:                      999
Time:                        23:07:56   Df Model:                            1
                               Mean Model                               
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu         5.6367e-03  7.036e-02  8.011e-02      0.936 [ -0.132,  0.144]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega          0.9549      2.241      0.426      0.670    [ -3.438,  5.347]
alpha[1]       0.0373  4.678e-02      0.798      0.425 [-5.437e-02,  0.129]
beta[1]        0.7617      0.502      1.517      0.129    [ -0.223,  1.746]
===========================================================================

Covariance estimator: robust
"""

Our `model` module is done! Now it's time to move on to the "main" course and add the final piece to our application.

# Main Module

Similar to the interactive applications we made in Projects 6 and 7, our first step here will be to create an `app` object. This time, instead of being a plotly application, it'll be a FastAPI application.

In [49]:
VimeoVideo("772219283", h="2cd1d97516", width=600)

**Task 8.4.9:** In the `main` module, instantiate a FastAPI application named `app`.

- [Instantiate an application in FastAPI.](../%40textbook/22-apis.ipynb#Creating-a-Path)

In order for our `app` to work, we need to run it on a server. In this case, we'll run the server on our virtual machine using the [uvicorn library](https://www.uvicorn.org/).

In [50]:
VimeoVideo("772219237", h="5ee74f82db", width=600)

**Task 8.4.10:** Go to the command line, navigate to the directory for this project, and start your app server by entering the following command.

```bash
uvicorn main:app --reload --workers 1 --host localhost --port 8008
```

Remember how the AlphaVantage API had a `"/query"` path that we accessed using a `get` HTTP request? We're going to build similar paths for our application. Let's start with an MVP example so we can learn how paths work in FastAPI. 

In [51]:
VimeoVideo("772219175", h="6f53c61020", width=600)

**Task 8.4.11:** Create a `"/hello"` path for your app that returns a greeting when it receives a `get` request.

- [Create an application path in FastAPI.](../%40textbook/22-apis.ipynb#Creating-a-Path)

We've got our path. Let's perform as `get` request to see if it works.

In [52]:
VimeoVideo("772219134", h="09a4b98413", width=600)

**Task 8.4.12:** Create a `get` request to hit the `"/hello"` path running at `"http://localhost:8008"`.

- [What's an HTTP request?](../%40textbook/22-apis.ipynb#RESTful-APIs)
- [Make an HTTP request using requests.](..//%40textbook/22-apis.ipynb#Making-an-HTTP-Request)

In [54]:
url = "http://localhost:8008/hello"
response = requests.get(url=url)

print("response code:", response.status_code)
response.json()

response code: 200


{'message': 'Hello world!'}

Excellent! Now let's start building the fun stuff.

## `"/fit"` Path

Our first path will allow the user to fit a model to stock data when they make a `post` request to our server. They'll have the choice to use new data from AlphaVantage, or older data that's already in our database. When a user makes a request, they'll receive a response telling them if the operation was successful or whether there was an error. 

One thing that's very important when building an API is making sure the user passes the correct parameters into the app. Otherwise, our app could crash! FastAPI works well with the [pydantic library](https://pydantic-docs.helpmanual.io/), which checks that each request has the correct parameters and data types. It does this by using special data classes that we need to define. Our `"/fit"` path will take user input and then output a response, so we need two classes: one for input and one for output.

In [55]:
VimeoVideo("772219078", h="4f016b11e1", width=600)

**Task 8.4.13:** Create definitions for a `FitIn` and a `FitOut` data class. The `FitIn` class should inherit from the pydantic `BaseClass`, and the `FitOut` class should inherit from the `FitIn` class. Be sure to include type hints.

- [Write a class definition in Python.](../%40textbook/21-python-object-oriented-programming.ipynb#Defining-a-Class)
- [What's class inheritance?](../%40textbook/21-python-object-oriented-programming.ipynb#Inheritance)
- [What are type hints?](../%40textbook/22-apis.ipynb#Data-Classes-and-Type-Checking)
- [Define a data model in pydantic.](../%40textbook/22-apis.ipynb#Data-Classes-and-Type-Checking)

With our data classes defined, let's see how pydantic ensures our that users are supplying the correct input and our application is returning the correct output.

In [56]:
VimeoVideo("772219008", h="ad1114eb9e", width=600)

**Task 8.4.14:** Use the code below to experiment with your `FitIn` and `FitOut` classes. Under what circumstances does instantiating them throw errors? What class or classes are they instances of?

- [What's class inheritance?](../%40textbook/21-python-object-oriented-programming.ipynb#Inheritance)
- [What are type hints?](../%40textbook/22-apis.ipynb#Data-Classes-and-Type-Checking)
- [Define a data model in pydantic.](../%40textbook/22-apis.ipynb#Data-Classes-and-Type-Checking)

In [59]:
from main import FitIn, FitOut

# Instantiate `FitIn`. Play with parameters.
fi = FitIn(
    ticker="SHOPERSTOP.BSE",
    use_new_data=True,
    n_observations=2000,
    p=1,
    q=1
)
print(fi)

# Instantiate `FitOut`. Play with parameters.
fo = FitOut(
    ticker="SHOPERSTOP.BSE",
    use_new_data=True,
    n_observations=2000,
    p=1,
    q=1,
    success=True,
    message = "Model is ready to rock!!"
)
print(fo)

ticker='SHOPERSTOP.BSE' use_new_data=True n_observations=2000 p=1 q=1
ticker='SHOPERSTOP.BSE' use_new_data=True n_observations=2000 p=1 q=1 success=True message='Model is ready to rock!!'


In [58]:
fi.schema

<bound method BaseModel.schema of <class 'main.FitIn'>>

One cool feature of FastAPI is that it can work in asynchronous scenarios. That's not something we need to learn for this project, but it does mean that we need to instantiate a `GarchModel` object each time a user makes a request. To make the coding easier for us, let's make a function to handle that process for us. 

In [60]:
VimeoVideo("772218958", h="37744c9d88", width=600)

**Task 8.4.15:** Create a `build_model` function in your `main` module. Use the docstring as a guide, and test your function below.

- [What's a function?](../%40textbook/02-python-advanced.ipynb#Functions)
- [Write a function in Python.](../%40textbook/02-python-advanced.ipynb#Functions)
- [What's an assert statement?](../%40textbook/02-python-advanced.ipynb#Testing-Code)
- [Write an assert statement in Python.](../%40textbook/02-python-advanced.ipynb#Testing-Code)

In [61]:
from main import build_model

# Instantiate `GarchModel` with function
model_shop = build_model(ticker="SHOPERSTOP.NS", use_new_data=False)

# Is `SQLRepository` attached to `model_shop`?
assert isinstance(model_shop.repo, SQLRepository)

# Is SQLite database attached to `SQLRepository`
assert isinstance(model_shop.repo.connection, sqlite3.Connection)

# Is `ticker` attribute correct?
assert model_shop.ticker == "SHOPERSTOP.NS"

# Is `use_new_data` attribute correct?
assert not model_shop.use_new_data

model_shop

We've got data classes, we've got a `build_model` function, and all that's left is to build the `"/fit"` path. We'll use our `"/hello"` path as a template, but we'll need to include more features, like error handling. 

In [62]:
VimeoVideo("772218892", h="6779ee3470", width=600)

**Task 8.4.16:** Create a `"/fit"` path for your `app`. It will take a `FitIn` object as input, and then build a `GarchModel` using the `build_model` function. The model will wrangle the needed data, fit to the data, and save the completed model. Finally, it will send a response in the form of a `FitOut` object. Be sure to handle any errors that may arise. 

- [Create an application path in FastAPI.](../%40textbook/22-apis.ipynb#Creating-a-Path)

Last step! Let's make a `post` request and see how our app responds.

In [63]:
VimeoVideo("772218833", h="6d27fb4539", width=600)

**Task 8.4.17:** Create a `post` request to hit the `"/fit"` path running at `"http://localhost:8008"`. You should train a GARCH(1,1) model on 2000 observations of the Shoppers Stop data you already downloaded. Pass in your parameters as a dictionary using the `json` argument.

- [What's an argument?](../%40textbook/01-python-getting-started.ipynb#JSON)
- [What's an HTTP request?](../%40textbook/22-apis.ipynb#RESTful-APIs)
- [Make an HTTP request using requests.](..//%40textbook/22-apis.ipynb#Making-an-HTTP-Request)

In [65]:
# URL of `/fit` path
url = "http://localhost:8008/fit"

# Data to send to path
json = {
    "ticker": "SHOPERSTOP.BSE",
     "use_new_data": False,
     "n_observations": 2000,
     "p": 1,
     "q": 1
}
# Response of post request
response = requests.post(url=url, json=json)
# Inspect response
print("response code:", response.status_code)
response.json()

response code: 200


{'ticker': 'SHOPERSTOP.BSE',
 'use_new_data': False,
 'n_observations': 2000,
 'p': 1,
 'q': 1,
 'success': True,
 'message': "Trained and saved 'models/2026-06-19T00:04:42.950118_SHOPERSTOP.BSE.pkl'."}

Boom! Now we can train models using the API we created. Up next: a path for making predictions. 

## `"/predict"` Path

For our `"/predict"` path, users will be able to make a `post` request with the ticker symbol they want a prediction for and the number of days they want to forecast into the future. Our app will return a forecast or, if there's an error, a message explaining the problem.

The setup will be very similar to our `"/fit"` path. We'll start with data classes for the in- and output.

In [66]:
VimeoVideo("772218808", h="3a73624069", width=600)

**Task 8.4.18:** Create definitions for a `PredictIn` and `PredictOut` data class. The `PredictIn` class should inherit from the pydantic `BaseModel`, and the `PredictOut` class should inherit from the `PredictIn` class. Be sure to include type hints. The use the code below to test your classes.

- [Write a class definition in Python.](../%40textbook/21-python-object-oriented-programming.ipynb#Defining-a-Class)
- [What's class inheritance?](../%40textbook/21-python-object-oriented-programming.ipynb#Inheritance)
- [What are type hints?](../%40textbook/22-apis.ipynb#Data-Classes-and-Type-Checking)
- [Define a data model in pydantic.](../%40textbook/22-apis.ipynb#Data-Classes-and-Type-Checking)

In [67]:
from main import PredictIn, PredictOut

pi = PredictIn(ticker="SHOPERSTOP.NS", n_days=5)
print(pi)

po = PredictOut(
    ticker="SHOPERSTOP.NS", n_days=5, success=True, forecast={}, message="success"
)
print(po)

ticker='SHOPERSTOP.NS' n_days=5
ticker='SHOPERSTOP.NS' n_days=5 success=True forecast={} message='success'


Up next, let's create the path. The good news is that we'll be able to reuse our `build_model` function.

In [68]:
VimeoVideo("772218740", h="ff06859ece", width=600)

**Task 8.4.19:** Create a `"/predict"` path for your `app`. It will take a `PredictIn` object as input, build a `GarchModel`, load the most recent trained model for the given ticker, and generate a dictionary of predictions. It will then return a `PredictOut` object with the predictions included. Be sure to handle any errors that may arise.

- [Create an application path in FastAPI.](../%40textbook/22-apis.ipynb#Creating-a-Path)

Last step, let's see what happens when we make a `post` request...

In [69]:
VimeoVideo("772218642", h="1da744b9e7", width=600)

**Task 8.4.20:** Create a `post` request to hit the `"/predict"` path running at `"http://localhost:8008"`. You should get the 5-day volatility forecast for Shoppers Stop. When you're satisfied, submit your work to the grader.

- [What's an HTTP request?](../%40textbook/22-apis.ipynb#RESTful-APIs)
- [Make an HTTP request using requests.](..//%40textbook/22-apis.ipynb#Making-an-HTTP-Request)

In [77]:
model_shop = GarchModel(
    ticker="SHOPERSTOP.NS",
    repo=repo,
    use_new_data=False
)

In [82]:
# URL of `/predict` path
url = "http://localhost:8008/predict"
# Data to send to path
json = {"ticker": "SHOPERSTOP.NS", "n_days": 5}
# Response of post request
response = requests.post(url=url, json=json)
# Response JSON to be submitted to grader
submission = response.json()
# Inspect JSON
submission

{'ticker': 'SHOPERSTOP.NS',
 'n_days': 5,
 'success': True,
 'forecast': {'2026-06-08T00:00:00': 2.1716239743470003,
  '2026-06-09T00:00:00': 2.17323998222548,
  '2026-06-10T00:00:00': 2.1745303216513565,
  '2026-06-11T00:00:00': 2.1755607626059357,
  '2026-06-12T00:00:00': 2.176383742103092},
 'message': ''}

We did it! Better said, **you** did it. You got data from the AlphaVantage API, you stored it in a SQL database, you built and trained a GARCH model to predict volatility, and you created your own API to serve predictions from your model. That's data engineering, data science, and model deployment all in one project. If you haven't already, now's a good time to give yourself a pat on the back. You definitely deserve it.

---
Copyright 2023 WorldQuant University. This
content is licensed solely for personal use. Redistribution or
publication of this material is strictly prohibited.
